<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Practicing with Docker in Duckietown

Docker packages software and the files it needs into images, then runs those images as containers. Duckietown uses containers to keep Duckiedrone services such as drivers, dashboards, and communication services separate and reproducible.

The previous notebook introduced containers, images, and the Docker daemon. Keeping services in separate containers lets us start, stop, and inspect each one individually.

Before starting to practice with these tools, we need to establish where our commands will take effect. A Docker command typed on your base station can act on a local Docker installation or on a daemon elsewhere, including the Duckiedrone. Before creating or removing containers, we need to distinguish our practice environment from the software that supports the robot.

In this notebook we will focus on setting up the correct space for practicing with Docker: choosing the terminal for the exercises, inspecting its Docker connection settings, and verifying that the local daemon responds. These checks prepare the environment for the hands-on activities that follow.

## Choose the terminal for Docker practice

Run the commands in this LX from a base-station terminal that has permission to use a local Docker Engine or Docker Desktop installation. For example, inside a [Duckietown Workspace](https://docs.duckietown.com/ente/duckietown-manual/10-setup/00-computer/setup-duckietown-workspace.html) terminal, or a [native Ubuntu environment](https://docs.duckietown.com/ente/duckietown-manual/10-setup/00-computer/setup-ubuntu.html) with [Docker installed as shown on the Duckietown manual](https://docs.duckietown.com/ente/duckietown-manual/10-setup/02-software/docker-installation.html). 

For local practice, use the base-station terminal rather than the LX editor terminal or an SSH shell on a Duckiedrone. Its Docker client should reach your local practice daemon, not a robot daemon or an unfamiliar remote or shared host.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
Base station
|
+-- LX VS Code editor
|     Read and edit the notebooks
|
+-- Base-station terminal
      Run Docker practice commands
                |
                v
      Local practice Docker daemon
                |
                v
      Exercise containers, images, and volumes
  </pre>
  <figcaption>
    Figure 1: Use the editor for the notebooks and a separate base-station terminal for Docker practice.
  </figcaption>
</figure>

## Identify the Docker target

Before a modifying command, confirm that its Docker target is your authorized local Docker host. `docker context show` prints the selected context name. Run it first:

```bash
docker context show
```

For example, the output might be:

```text
default
```

List all the configured contexts and their connection endpoints:

```bash
docker context ls
```

An illustrative Ubuntu configuration is:

```text
NAME        DESCRIPTION                               DOCKER ENDPOINT               ERROR
default *   Current DOCKER_HOST based configuration    unix:///var/run/docker.sock
```

The `*` marks the selected context. The `DOCKER ENDPOINT` column shows its connection address. Here, `unix:///var/run/docker.sock` identifies a Unix socket, a mechanism for communication between processes on the same system.

These commands inspect the client’s configuration. Docker Desktop or a different local installation may show another context name and socket path. Compare your result with the setup instructions for your installation; do not switch contexts simply to make the output match this example. See [Docker contexts](https://docs.docker.com/engine/manage-resources/contexts/) for additional detail.

### Check for environment overrides

The Linux and Networking LX introduced environment variables as settings inherited by programs started from a shell. Docker uses them too.

Two variables can affect which daemon receives your commands:

- `DOCKER_HOST` specifies a daemon endpoint.
- `DOCKER_CONTEXT` selects a context and takes precedence over `DOCKER_HOST`.

Display their values in the same terminal. The expression `${VARIABLE:-<unset>}` prints `<unset>` when the variable is absent or empty:

```bash
printf 'DOCKER_HOST=%s\n' "${DOCKER_HOST:-<unset>}"
printf 'DOCKER_CONTEXT=%s\n' "${DOCKER_CONTEXT:-<unset>}"
```

In this example, neither variable supplies an override.

```text
DOCKER_HOST=<unset>
DOCKER_CONTEXT=<unset>
```

If either contains a value, establish what it selects before continuing. For example, a `DOCKER_CONTEXT` value naming a robot connection would make that terminal unsuitable for local practice. Do not clear unfamiliar settings or guess another endpoint. Check the installation’s documented configuration or ask the person who manages it.

## Verify that the daemon responds

Once the context and any environment overrides identify your authorized local Docker installation, run:

```bash
docker version
```

This command reports information about both the client and the daemon, called the **server** in its output. For example, output from a Duckietown Workspace might look like this:

```text
ubuntu@orbstack:~$ docker version
Client:
 Version:           28.5.2-1
 API version:       1.51
 Go version:        go1.24.9
 Git commit:        ecc694264de6b34e4b59d16245603382f22fa813
 Built:             Wed Nov  5 10:32:31 UTC 2025
 OS/Arch:           linux/arm64
 Context:           default

Server:
 Engine:
  Version:          28.5.2-1
  API version:      1.51 (minimum version 1.24)
  Go version:       go1.24.9
  Git commit:       89c5e8fd66634b6128fc4c0e6f1236e2540e46e0
  Built:            Wed Nov  5 14:19:32 2025
  OS/Arch:          linux/arm64
  Experimental:     false
 containerd:
  Version:          2.3.4-2
  GitCommit:        db8809540e1a7a9da5d518876894933ff55692ab
 runc:
  Version:          1.5.1-9
  GitCommit:        8f2685a471d3347a686ad3909783d8aafc6bb208
 docker-init:
  Version:          0.19.0
  GitCommit:        
```

The two sections answer different questions:

| Output section | What it tells you |
| --- | --- |
| `Client` | Which Docker command-line client is running in this terminal |
| `Server` | Which Docker Engine answered the request |

A successful response containing both sections shows that the client can communicate with the selected daemon. It does not independently prove that you selected the intended machine; that is why we checked the connection settings first.

`OS/Arch` reports the operating system and processor architecture of each component. With Docker Desktop on an Apple Silicon Mac, for example, the client may report `darwin/arm64` while the server reports `linux/arm64`. The client runs on macOS, and the Linux daemon runs inside Docker Desktop’s virtual machine.

Note the difference between `docker version` and `docker --version`: the latter reports the client version only. See the [`docker version` reference](https://docs.docker.com/reference/cli/docker/version/).

### If the check fails

Use the message to identify which part needs attention:

| Observation | What to check |
| --- | --- |
| `docker: command not found` | Whether the Docker client is installed and available in this terminal |
| Client information followed by a connection error | Whether the intended local daemon is running and its endpoint is configured correctly |
| A permission-denied error | Whether your account has the access required by the documented local setup |

If you use Docker Desktop, check that it has finished starting. For a managed installation, follow the administrator’s instructions.

In general, the learning activities in this LX do not require `sudo`, `--privileged`, `--device`, `-H`, or a mount of `/var/run/docker.sock`, unless explicitly mentioned. Before any command that creates, changes, or removes resources, confirm that you are still using the verified local practice connection.

## Keep practice resources identifiable

Your local Docker installation may also support other work, including the LX editor itself. We therefore need to distinguish exercise resources from resources that were already present.

Containers, images, and volumes created specifically for these exercises will use names beginning with `lx-docker-`. For example:

```text
lx-docker-camera-a
```
The prefix helps you recognize an exercise resource, but it does not establish ownership by itself. Remove only resources that you created for this LX. If an exercise name is already in use, inspect the situation before proceeding rather than deleting the existing resource.

We will also remove resources individually by name. Broad cleanup commands such as `docker system prune`, `docker container prune`, `docker image prune`, and `docker volume prune` can affect work outside the current exercise, so they are not used here.

## Connect the checks to the next activities

The previous notebook distinguished an image from a container. Keep that distinction in mind when interpreting the upcoming exercises:

| Action | Expected result |
| --- | --- |
| Create two containers from one image | Two instances with separate writable layers |
| Stop a container | Its processes stop; its writable layer remains |
| Remove a container | Its writable layer is removed |
| Run a temporary container with `--rm` | Docker removes it automatically after it exits |

A camera-processing result stored only in a container’s writable layer would therefore survive stopping that container, but not removing it. Later, we will use persistent storage to keep results independently of the container.

The next notebooks put these ideas into practice:

- [Notebook 5](./5-run-and-inspect-containers.ipynb) introduces running, inspecting, entering, and removing a named container.
- [Notebook 6](./6-build-and-test-a-local-image.ipynb) builds a small web-server image and tests it through a port published only on the base station’s loopback address.
- [Notebook 7](./7-docker-volumes-bind-mounts-and-cleanup.ipynb) explores persistent data using a named volume and a read-only bind mount.

## Further reading

Docker’s [context documentation](https://docs.docker.com/engine/manage-resources/contexts/) explains how clients select daemons. The [`docker version` reference](https://docs.docker.com/reference/cli/docker/version/) describes the client and server information used in this notebook.

## Checkpoint

Before running the self-check, record:

1. Which terminal you will use for Docker practice.
2. Your selected Docker context and its endpoint.
3. Whether `DOCKER_HOST` or `DOCKER_CONTEXT` has a value.
4. Whether `docker version` returned both client and server information.

Then consider: if `docker --version` succeeds, but `docker version` reports a connection error, which component have you confirmed is available?

Run the self-check in the next cell. Write or select a response before revealing the answer.

In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

You have identified a terminal and local Docker connection for practice, checked for environment overrides, and confirmed whether the daemon responds. The `lx-docker-` naming convention will help you recognize your exercise resources. Next, [read image references and inspect the local inventory](./3-docker-clients-registries-and-platforms.ipynb).